# Лабораторная работа по NLP №2

Выполнили студенты:
- Кудасов Максим, 21ПМИ-2
- Красильников Николай, 21ПМИ-1
- Ерёменко Даниил, 21ПМИ-1

## Подготовка

In [ ]:
import os
import logging
import math
from datetime import datetime

from transformers import (
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    AutoTokenizer,
    TrainerCallback,
)
from datasets import Dataset
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

os.environ["TOKENIZERS_PARALLELISM"] = "true"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.FileHandler("training.log"), logging.StreamHandler()],
)
logger = logging.getLogger(__name__)

In [ ]:
class TextDataset(Dataset):
    def __init__(self, text, chunk_len=200, stride=50):
        self.text = text
        self.chunk_len = chunk_len
        self.stride = stride
        self.unique_chars = sorted(set(text))
        self.char_to_idx = {c: i for i, c in enumerate(self.unique_chars)}
        self.idx_to_char = {i: c for i, c in enumerate(self.unique_chars)}
        self.data = self._process_text()

    def __len__(self):
        return len(self.data)

    def _process_text(self):
        sequences = []
        for i in range(0, len(self.text) - self.chunk_len, self.stride):
            chunk = self.text[i:i+self.chunk_len+1]
            sequences.append(chunk)
        return sequences

    def __getitem__(self, idx):
        chunk = self.data[idx]
        input_seq = [self.char_to_idx[c] for c in chunk[:-1]]
        target_seq = [self.char_to_idx[c] for c in chunk[1:]]
        return torch.LongTensor(input_seq), torch.LongTensor(target_seq)

    @property
    def vocab_size(self):
        return len(self.unique_chars)

### Загрузка датасета с arxiv

### Загрузка датасета с РИА Новости

### RNN

In [ ]:
class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, model="gru", n_layers=1, dropout=0.2):
        super().__init__()
        self.model_type = model.lower()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.n_layers = n_layers

        self.encoder = nn.Embedding(input_size, hidden_size)
        rnn_class = nn.GRU if self.model_type == "gru" else nn.LSTM
        self.rnn = rnn_class(
            hidden_size, hidden_size, n_layers,
            dropout=dropout if n_layers > 1 else 0,
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout)
        self.decoder = nn.Linear(hidden_size, output_size)

    def forward(self, input, hidden=None):
        batch_size = input.size(0)
        encoded = self.encoder(input)
        output, hidden = self.rnn(encoded, hidden)
        output = self.dropout(output)
        decoded = self.decoder(output.contiguous().view(-1, self.hidden_size))
        return decoded.view(batch_size, -1, self.output_size), hidden

    def init_hidden(self, batch_size, device):
        if self.model_type == "lstm":
            return (
                torch.zeros(self.n_layers, batch_size, self.hidden_size).to(device),
                torch.zeros(self.n_layers, batch_size, self.hidden_size).to(device)
            )
        else:
            return torch.zeros(self.n_layers, batch_size, self.hidden_size).to(device)

In [ ]:
def generate_sample(model, dataset, device, prompt="The", max_length=500, temperature=1.0, top_k=10, top_p=0.9):
    model.eval()
    generated = []
    input_seq = torch.LongTensor([dataset.char_to_idx[c] for c in prompt]).unsqueeze(0).to(device)  # (batch=1, seq_len)
    hidden = model.init_hidden(1, device)

    with torch.no_grad():
        if len(prompt) > 0:
            _, hidden = model(input_seq, hidden)

        input_seq = input_seq[:, -1].unsqueeze(1)

        for _ in range(max_length):
            outputs, hidden = model(input_seq, hidden)
            logits = outputs[:, -1, :] / temperature  # Берем последний выходной токен

            if top_k > 0:
                logits = _top_k_filter(logits, top_k)
            if top_p > 0.0:
                logits = _top_p_filter(logits, top_p)

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            generated.append(next_token.item())
            input_seq = next_token

    generated_str = prompt + ''.join([dataset.idx_to_char[idx] for idx in generated])
    print("\nGenerated text:")
    print(generated_str)
    return generated_str

def _top_k_filter(logits, k):
    values, _ = torch.topk(logits, k)
    min_values = values[:, -1].unsqueeze(1)
    return torch.where(logits < min_values, torch.ones_like(logits)*-float('inf'), logits)

def _top_p_filter(logits, p):
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

    sorted_indices_to_remove = cumulative_probs > p
    sorted_indices_to_remove[..., 0] = 0
    indices_to_remove = sorted_indices_to_remove.scatter(
        1, sorted_indices, sorted_indices_to_remove
    )
    return logits.masked_fill(indices_to_remove, -float('inf'))

In [ ]:
def evaluate(model, val_loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            hidden = model.init_hidden(inputs.size(0), device)
            outputs, _ = model(inputs, hidden)
            loss = criterion(outputs.transpose(1, 2), targets)
            total_loss += loss.item() * inputs.size(0)
    return total_loss / len(val_loader.dataset)

def train_model(model, dataset, epochs=50, batch_size=32, lr=3e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True
    )

    optimizer = torch.optim.AdamW([
        {'params': model.encoder.parameters(), 'weight_decay': 0.01},
        {'params': model.rnn.parameters()},
        {'params': model.decoder.parameters(), 'weight_decay': 0.01}
    ], lr=lr, fused=True)

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=lr,
        total_steps=epochs * len(loader),
        pct_start=0.1
    )

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = torch.amp.GradScaler()

    writer = SummaryWriter(
        log_dir=f"runs/LR_{lr:.6f}-model_type_{model.model_type}-hidden_size_{model.hidden_size}-n_layers_{model.n_layers}-batch_size_{batch_size}"
        )

    best_loss = float('inf')
    grad_norms = []

    max_grad_norm = 1.0
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        progress = tqdm(loader, desc=f"Epoch {epoch+1}", leave=False)

        for batch_idx, (inputs, targets) in enumerate(progress):
            current_batch_size = inputs.size(0)
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                hidden = model.init_hidden(current_batch_size, device)
                outputs, _ = model(inputs, hidden)
                loss = criterion(outputs.transpose(1, 2), targets)
                l2_reg = sum(p.norm(2) for p in model.parameters())
                loss += 0.001 * l2_reg

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=max_grad_norm,
                norm_type=2,
                error_if_nonfinite=False
            )

            grad_norms.append(grad_norm.item())

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_loss += loss.item()
            progress.set_postfix({
                'loss': f"{loss.item():.4f}",
                'grad': f"{grad_norm:.2f}",
                'lr': f"{optimizer.param_groups[0]['lr']:.2e}"
            })

            if batch_idx % 10 == 0:
                writer.add_scalar('Train/Loss', loss.item(), epoch*len(loader)+batch_idx)
                writer.add_scalar('Train/Grad_Norm', grad_norm.item(), epoch*len(loader)+batch_idx)
                writer.add_scalar('LR', optimizer.param_groups[0]['lr'], epoch*len(loader)+batch_idx)

        avg_loss = total_loss / len(loader)
        writer.add_scalar('Epoch/Loss', avg_loss, epoch)

        logging.info(f"Epoch {epoch+1}/{epochs} - "
                    f"Loss: {avg_loss:.4f} - "
                    f"Grad Norm: {grad_norm:.2f} - "
                    f"LR: {optimizer.param_groups[0]['lr']:.2e}")

        if avg_loss < best_loss and not torch.isnan(torch.tensor(avg_loss)):
            best_loss = avg_loss
            torch.save({
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'epoch': epoch,
                'loss': avg_loss
            }, "best_model.pth")

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    writer.close()
    return best_loss

### HF Transformer

In [ ]:
class RussianGPT:
    def __init__(self, config):
        self.config = config
        self.model = None
        self.tokenizer = None
        self.dataset = None

        # Create output directories
        os.makedirs(self.config["output_dir"], exist_ok=True)
        os.makedirs(self.config["log_dir"], exist_ok=True)

    def prepare_data(self):
        """Load and process training data into chunks without splitting"""
        logger.info("Loading and chunking data...")
        df = pd.read_json(self.config["data_path"])
        full_text = "\n".join(df["title"] + " " + df["text"])
        chunked_text = [
            full_text[i : i + self.config["chunk_size"]]
            for i in range(0, len(full_text), self.config["chunk_size"])
        ]
        self.dataset = Dataset.from_dict({"text": chunked_text})
        return self

    def split_dataset(self):
        """Split the dataset into train/test"""
        logger.info("Splitting dataset...")
        split_dataset = self.dataset.train_test_split(
            test_size=self.config["test_size"], shuffle=True, seed=42
        )
        self.dataset = split_dataset
        return self

    def load_tokenizer(self):
        """Load pre-trained tokenizer using AutoTokenizer with fast=True"""
        logger.info("Loading pre-trained tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.config["model_checkpoint"], use_fast=True
        )
        return self

    def initialize_model(self):
        """Load pre-trained model from checkpoint"""
        logger.info("Loading pre-trained model...")
        self.model = AutoModelForCausalLM.from_pretrained(self.config["model_checkpoint"])
        logger.info(f"Model loaded with {self.model.num_parameters() / 1e6:.2f}M parameters")
        return self

    def tokenize_dataset(self):
        """Tokenize both train and test splits using the pre-trained tokenizer"""
        logger.info("Tokenizing dataset...")

        def tokenize_fn(examples):
            return self.tokenizer(
                examples["text"],
                truncation=True,
                max_length=self.config["n_positions"],
                padding="max_length",
                add_special_tokens=True,
            )

        self.dataset = self.dataset.map(
            tokenize_fn, batched=True, num_proc=1, remove_columns=["text"]
        )
        return self

    def train(self):
        """Execute full training pipeline (fine-tuning)"""
        logger.info(f"Dataset features: {self.dataset}")
        torch.cuda.empty_cache()

        training_args = TrainingArguments(
            output_dir=self.config["output_dir"],
            logging_dir=self.config["log_dir"],
            num_train_epochs=self.config["num_epochs"],
            per_device_train_batch_size=self.config["batch_size"],
            per_device_eval_batch_size=self.config["batch_size"] // 2,
            gradient_accumulation_steps=self.config["grad_accum_steps"],
            learning_rate=self.config["learning_rate"],
            weight_decay=0.1,
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=100,
            save_steps=250,
            eval_steps=250,
            eval_strategy="steps",
            optim="adamw_torch_fused",
            gradient_checkpointing=True,
            report_to=["tensorboard"],
            dataloader_num_workers=4,
            torch_compile=True,
            resume_from_checkpoint=True,
            save_total_limit=2,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            max_grad_norm=1.0,
            warmup_steps=250,
            lr_scheduler_type="cosine",
        )

        # Создаём callback-и
        training_callback = TrainingProgressCallback()
        save_tokenizer_callback = SaveTokenizerCallback()
        inference_callback = InferenceLoggingCallback(self.config.get("eval_prompt", "Пример инференса"))

        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=self.dataset["train"],
            eval_dataset=self.dataset["test"],
            data_collator=DataCollatorForLanguageModeling(
                tokenizer=self.tokenizer, mlm=False
            ),
            callbacks=[training_callback, save_tokenizer_callback, inference_callback],
        )

        # Передаём ссылку на trainer в callback для инференса
        inference_callback.trainer = trainer

        logging.info(
            f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB"
        )
        logger.info("Starting training...")
        trainer.train()
        logger.info("Training completed. Saving final model...")

        self.model.save_pretrained(os.path.join(self.config["output_dir"], "final_model"))
        self.tokenizer.save_pretrained(os.path.join(self.config["output_dir"], "final_model"))
        return self


class TrainingProgressCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if state.is_local_process_zero:
            loss = logs.get("loss", float("nan"))
            eval_loss = logs.get("eval_loss", float("nan"))
            lr = logs.get("learning_rate", float("nan"))
            loss_str = f"{loss:.4f}" if not math.isnan(loss) else "N/A"
            eval_str = f"{eval_loss:.4f}" if not math.isnan(eval_loss) else "N/A"
            lr_str = f"{lr:.2e}" if not math.isnan(lr) else "N/A"
            logger.info(
                f"Step {state.global_step} | Loss: {loss_str} | Eval Loss: {eval_str} | Learning Rate: {lr_str}"
            )


class SaveTokenizerCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        trainer = kwargs.get("trainer", None)
        if trainer is not None and hasattr(trainer.data_collator, "tokenizer") and trainer.data_collator.tokenizer is not None:
            tokenizer_save_path = os.path.join(args.output_dir, f"checkpoint-{state.global_step}")
            trainer.data_collator.tokenizer.save_pretrained(tokenizer_save_path)
            logger.info(f"Tokenizer saved at {tokenizer_save_path}")
        return control


class InferenceLoggingCallback(TrainerCallback):
    """
    Callback для логирования инференс-вывода каждые eval_steps.
    Использует фиксированный prompt, задаваемый в конфигурации (ключ 'eval_prompt').
    """
    def __init__(self, eval_prompt: str):
        self.eval_prompt = eval_prompt
        self.trainer = None  # Будет установлен вручную

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if self.trainer is None:
            logger.warning("Trainer не найден в on_evaluate")
            return control

        # Используем токенизатор из data_collator, так как атрибут tokenizer уже deprecated
        if not hasattr(self.trainer.data_collator, "tokenizer") or self.trainer.data_collator.tokenizer is None:
            logger.warning("Tokenizer не найден в trainer.data_collator")
            return control

        tokenizer = self.trainer.data_collator.tokenizer
        inputs = tokenizer(self.eval_prompt, return_tensors="pt")
        inputs = {k: v.to(self.trainer.model.device) for k, v in inputs.items()}

        generated_ids = self.trainer.model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=True,
            temperature=1.0,
            top_p=0.95,
            repetition_penalty=1.0,
        )
        generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
        logger.info(
            f"Inference at step {state.global_step} | Prompt: '{self.eval_prompt}' | Generated: {generated_text}"
        )
        return control

## Ход работы

### 1. Тренировка RNN на arxiv

In [ ]:
def train_best_model(params):
	df = pd.read_csv('data/arxiv.csv')
	text = ' '.join(df['summary'].dropna().values)
	dataset = TextDataset(text, chunk_len=250, stride=100)

	model = CharRNN(
		input_size=dataset.vocab_size,
		hidden_size=params['hidden_size'],
		output_size=dataset.vocab_size,
		model=params['model_type'],
		n_layers=params['n_layers'],
		dropout=params['dropout']
	)

	best_loss = train_model(
		model,
		dataset,
		epochs=params['epochs'],
		batch_size=params['batch_size'],
		lr=params['lr'],
	)

	return model, best_loss

In [ ]:
best_params = {
	'model_type': 'lstm',
	'hidden_size': 474,
	'n_layers': 1,
	'dropout': 0.348681,
	'lr': 0.003438,
	'batch_size': 64,
	'epochs': 25,
}

best_model, best_loss = train_best_model(best_params)

### 2. Тренировка RNN на РИА Новости

### 3. Fine-Tuning Transformer на РИА Новости

In [ ]:
def inference(
    prompt: str,
    model_dir: str,
    n: int = 1,
    repetition_penalty: float = 1.0,
    temperature: float = 1.0,
    top_p: float = 1.0,
    top_k: int = -1,
    max_tokens: int = 50,
    min_tokens: int = 0,
    skip_special_tokens: bool = True,
):
    # Загружаем токенайзер и модель из указанной директории
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForCausalLM.from_pretrained(model_dir)

    # Переводим модель на нужное устройство (GPU, если доступен)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Токенизируем входной текст
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {key: val.to(device) for key, val in inputs.items()}

    # Если top_k равен -1, отключаем top-k фильтрацию
    if top_k <= 0:
        top_k = None

    # Вычисляем параметры генерации.
    # Используем max_new_tokens и min_new_tokens, чтобы задать число генерируемых токенов
    do_sample = temperature > 0.0  # при temperature == 0 происходит жадное декодирование

    generate_kwargs = {
        "do_sample": do_sample,
        "num_return_sequences": n,
        "repetition_penalty": repetition_penalty,
        "top_p": top_p,
        "max_new_tokens": max_tokens,
        "min_new_tokens": min_tokens,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if do_sample:
        generate_kwargs["temperature"] = temperature
    if top_k is not None:
        generate_kwargs["top_k"] = top_k

    # Генерация последовательностей
    outputs = model.generate(**inputs, **generate_kwargs)

    # Декодируем результаты
    responses = [
        tokenizer.decode(output, skip_special_tokens=skip_special_tokens)
        for output in outputs
    ]
    return responses

In [ ]:
config = {
    "data_path": "data/ria_articles.json",
    "test_size": 0.05,
    "output_dir": f"./models/rugpt-{datetime.now().strftime('%Y%m%d-%H%M')}",
    "log_dir": "./logs",
    "chunk_size": 1024,
    "n_positions": 1024,
    "num_epochs": 5,
    "batch_size": 16,
    "grad_accum_steps": 4,
    "learning_rate": 5e-4,
    "eval_prompt": "Путин сказал:",
    # Контрольный пункт предобученной модели (русскоязычная модель)
    "model_checkpoint": "sberbank-ai/rugpt3small_based_on_gpt2",
}

In [ ]:
(
	RussianGPT(config)
	.prepare_data()
	.load_tokenizer()
	.split_dataset()
	.initialize_model()
	.tokenize_dataset()
	.train()
)

### Выводы по ходу работы